# Data Handling

Before training, we need to load and organize the data.

![PyTorch data-loading pipeline](diagrams/04-data-handling.svg)

## Load the Animals Dataset

We predict `type` using seven animal features.

- Bird = `0`
- Mammal = `1`
- Reptile = `2`

`animal_name` is not a feature because the name does not describe the animal's properties.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

# This works when the notebook starts from either the notes folder or project root
csv_path = Path("animal_dataset.csv")
if not csv_path.exists():
    csv_path = Path("notes/animal_dataset.csv")

animals = pd.read_csv(csv_path)  # CSV file -> pandas DataFrame
animals

## Define Features and Targets

```text
animal_name | hair ... tail | type
   remove    |      X        |  y
```

- `X` contains the input features.
- `y` contains the correct class for each row.

In [ ]:
# All rows, excluding the first and last columns
features = animals.iloc[:, 1:-1]
X = features.to_numpy(dtype=np.float32)  # model inputs need floats

# All rows from the last column
y = animals.iloc[:, -1].to_numpy(dtype=np.int64)  # class labels need integers

print(f"X shape: {X.shape} -> 5 animals, 7 features each")
print(X)
print(f"\ny shape: {y.shape} -> 1 target class per animal")
print(y)

## TensorDataset

`TensorDataset` keeps each input row connected to its target.

```text
X_tensor[0] + y_tensor[0] -> dataset[0]
```

In [ ]:
X_tensor = torch.tensor(X, dtype=torch.float32)  # features for the model
y_tensor = torch.tensor(y, dtype=torch.long)     # targets for CrossEntropyLoss

dataset = TensorDataset(X_tensor, y_tensor)

# dataset[0] returns the features and target from the same row
input_sample, label_sample = dataset[0]

print(f"Number of samples: {len(dataset)}")
print(f"First input: {input_sample}")
print(f"First label: {label_sample} -> bird")

## DataLoader

`DataLoader` gives the model small batches instead of the entire dataset at once.

- `batch_size=2`: return two animals per iteration
- `shuffle=True`: change the row order each epoch
- **Epoch**: one full pass through the dataset
- **Generalization**: performing well on new data, not only memorizing training data

In [ ]:
generator = torch.Generator().manual_seed(42)  # repeatable shuffle for this example

dataloader = DataLoader(
    dataset,
    batch_size=2,  # number of samples returned per iteration
    shuffle=True,  # shuffle samples at the start of each epoch
    generator=generator,
)

print(f"Number of batches: {len(dataloader)}")

## Iterate Through Batches

There are five animals and two animals per batch:

```text
Batch 1: 2 animals
Batch 2: 2 animals
Batch 3: 1 animal
```

In [ ]:
for batch_number, (batch_inputs, batch_labels) in enumerate(dataloader, start=1):
    # batch_inputs and batch_labels contain matching rows
    print(f"Batch {batch_number}")
    print(f"  Input shape: {batch_inputs.shape}")
    print(f"  Labels: {batch_labels}")

## Quick Look

```python
animals = pd.read_csv("animal_dataset.csv")

X = animals.iloc[:, 1:-1].to_numpy(dtype=np.float32)
y = animals.iloc[:, -1].to_numpy(dtype=np.int64)

dataset = TensorDataset(torch.tensor(X), torch.tensor(y))
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

for batch_inputs, batch_labels in dataloader:
    # Forward pass, loss, and backpropagation will go here
    pass
```